In [ ]:
# ============================================================
# MODEL 1 — Retailer Priority Scoring
# Training + Saving + Prediction Pipeline
#
# Tech:
# - pandas
# - scikit-learn
# - xgboost
# - joblib
#
# Output:
# - priority_model.pkl
#
# ============================================================

import pandas as pd
import numpy as np
import joblib

from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder


# ============================================================
# 1. LOAD DATA
# ============================================================

retailer_pos = pd.read_csv("retailer_pos.csv")
inventory = pd.read_csv("retailer_inventory_weekly.csv")
visit_log = pd.read_csv("retailer_visit_log.csv")
retailers = pd.read_csv("retailers.csv")
whatsapp = pd.read_csv("whatsapp_message_log.csv")
growers = pd.read_csv("growers.csv")


# ============================================================
# 2. DATE CLEANING
# ============================================================

retailer_pos["transaction_date"] = pd.to_datetime(
    retailer_pos["transaction_date"]
)

inventory["week_end_date"] = pd.to_datetime(
    inventory["week_end_date"]
)

visit_log["visit_date"] = pd.to_datetime(
    visit_log["visit_date"]
)

whatsapp["message_sent_date"] = pd.to_datetime(
    whatsapp["message_sent_date"]
)


# ============================================================
# 3. FEATURE ENGINEERING
# ============================================================

# ------------------------------------------------------------
# SALES FEATURES
# ------------------------------------------------------------

sales_features = (
    retailer_pos
    .groupby("retailer_id")
    .agg(
        total_sales_qty=("sku_qty", "sum"),
        total_revenue=(
            "sku_price",
            "sum"
        ),
        avg_sales_qty=("sku_qty", "mean"),
        sales_transactions=("transaction_id", "count")
    )
    .reset_index()
)

# ------------------------------------------------------------
# INVENTORY FEATURES
# ------------------------------------------------------------

inventory_features = (
    inventory
    .groupby("retailer_id")
    .agg(
        avg_inventory=("sku_qty", "mean"),
        stockout_frequency=(
            "sku_qty",
            lambda x: (x == 0).sum()
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# VISIT FEATURES
# ------------------------------------------------------------

latest_date = visit_log["visit_date"].max()

visit_features = (
    visit_log
    .groupby("territory_id")
    .agg(
        total_visits=("visit_date", "count"),
        last_visit=("visit_date", "max")
    )
    .reset_index()
)

visit_features["days_since_last_visit"] = (
    latest_date - visit_features["last_visit"]
).dt.days

visit_features.drop(columns=["last_visit"], inplace=True)

# ------------------------------------------------------------
# WHATSAPP ENGAGEMENT
# ------------------------------------------------------------

whatsapp_features = (
    whatsapp
    .groupby("grower_id")
    .agg(
        opened_rate=("opened_status", "mean"),
        clicked_rate=("clicked_status", "mean")
    )
    .reset_index()
)

# ------------------------------------------------------------
# GROWER FEATURES
# ------------------------------------------------------------

grower_features = (
    growers
    .groupby(["district", "tehsil"])
    .agg(
        avg_farm_size=("grower_farm_size", "mean"),
        avg_grower_age=("grower_age", "mean"),
        product_scan_rate=("product_scan", "mean")
    )
    .reset_index()
)

# ============================================================
# 4. MERGE FEATURES
# ============================================================

df = retailers.copy()

# Merge sales
df = df.merge(
    sales_features,
    on="retailer_id",
    how="left"
)

# Merge inventory
df = df.merge(
    inventory_features,
    on="retailer_id",
    how="left"
)

# Merge visits
df = df.merge(
    visit_features,
    on="territory_id",
    how="left"
)

# Merge grower data
df = df.merge(
    grower_features,
    on=["district", "tehsil"],
    how="left"
)

# Fill missing values
df.fillna(0, inplace=True)


# ============================================================
# 5. CREATE TARGET LABEL
#
# Priority Score = business value score
# ============================================================

df["priority_score"] = (
    0.35 * df["total_sales_qty"] +
    0.25 * df["stockout_frequency"] +
    0.20 * df["sales_transactions"] +
    0.10 * df["days_since_last_visit"] +
    0.10 * df["product_scan_rate"]
)

# Normalize to 0-100
min_score = df["priority_score"].min()
max_score = df["priority_score"].max()

df["priority_score"] = (
    (
        (df["priority_score"] - min_score)
        /
        (max_score - min_score)
    ) * 100
)

# ============================================================
# 6. ENCODE CATEGORICAL FEATURES
# ============================================================

label_encoders = {}

categorical_columns = [
    "state",
    "district",
    "tehsil"
]

for col in categorical_columns:

    le = LabelEncoder()

    df[col] = le.fit_transform(df[col])

    label_encoders[col] = le


# ============================================================
# 7. SELECT FEATURES
# ============================================================

feature_columns = [
    "state",
    "district",
    "tehsil",
    "total_sales_qty",
    "total_revenue",
    "avg_sales_qty",
    "sales_transactions",
    "avg_inventory",
    "stockout_frequency",
    "total_visits",
    "days_since_last_visit",
    "avg_farm_size",
    "avg_grower_age",
    "product_scan_rate"
]

X = df[feature_columns]

y = df["priority_score"]


# ============================================================
# 8. TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# ============================================================
# 9. TRAIN MODEL
# ============================================================

model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)


# ============================================================
# 10. EVALUATE
# ============================================================

preds = model.predict(X_test)

mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)

print("\n========================")
print("MODEL PERFORMANCE")
print("========================")
print(f"MAE : {mae:.2f}")
print(f"R2  : {r2:.2f}")


# ============================================================
# 11. SAVE MODEL
# ============================================================

joblib.dump(model, "priority_model.pkl")

joblib.dump(label_encoders, "label_encoders.pkl")

joblib.dump(feature_columns, "feature_columns.pkl")

print("\nModel saved successfully!")


# ============================================================
# 12. SAMPLE PREDICTION
# ============================================================

sample_input = pd.DataFrame([{
    "state": "Uttar Pradesh",
    "district": "Lucknow",
    "tehsil": "Malihabad",
    "total_sales_qty": 120,
    "total_revenue": 55000,
    "avg_sales_qty": 12,
    "sales_transactions": 45,
    "avg_inventory": 18,
    "stockout_frequency": 3,
    "total_visits": 15,
    "days_since_last_visit": 8,
    "avg_farm_size": 4.5,
    "avg_grower_age": 47,
    "product_scan_rate": 0.62
}])


# ============================================================
# 13. APPLY LABEL ENCODING
# ============================================================

for col in categorical_columns:

    le = label_encoders[col]

    sample_input[col] = le.transform(
        sample_input[col]
    )


# ============================================================
# 14. LOAD MODEL
# ============================================================

loaded_model = joblib.load("priority_model.pkl")


# ============================================================
# 15. PREDICT
# ============================================================

prediction = loaded_model.predict(sample_input)

print("\n========================")
print("PREDICTION")
print("========================")

print(
    f"Retailer Priority Score: {prediction[0]:.2f}"
)